In [1]:
import os
import json
from pathlib import Path

import requests
import pandas as pd


from dw_helpers import ensure_season_charts, publish, BASE_URL
from dw_helpers import headers as dw_headers
from fantrax_helpers import resolve_latest_gw, fetch_standings, fetch_matchups

In [2]:
# Pick a season with SEASON (e.g. `make update SEASON=16`); default is the newest in seasons.json.
# To start a new season, add its Fantrax league ID to seasons.json -- that's it.
SEASONS = json.loads(Path('seasons.json').read_text())
season = os.environ.get('SEASON', '').strip() or max(SEASONS, key=int)
league_id = SEASONS[season]['league_id']
template_season = str(int(season) - 1)  # a new season's Datawrapper charts are copied from last season's

# Auto-detects the last gameweek Fantrax has scored. To force one, set LATEST_GW,
# e.g. `make update LATEST_GW=10`
latest_gw = resolve_latest_gw(league_id)
print(f'Season {season}, league {league_id}')

latest_gw = 11 (auto-detected)
Season 16, league ag5qep8dms84zona


In [3]:
# All Fantrax requests live in fantrax_helpers.py
df, team_info = fetch_standings(league_id, latest_gw)

In [4]:
df.shortName.unique()

array(['CFC', 'NPSG', 'WTU', 'ZAC', 'RAFA', 'Champion', 'BFC', 'NUF🖕🏾',
       'SHFC', 'THOTS'], dtype=object)

In [5]:
df_trim = df[
    df.gw <= latest_gw
][['W', 'D', 'L','Points','Win%', 'WW', 'FPtsF', 'FPtsA', 'Streak','team','rank','gw']]

In [6]:
# data = df_trim

# chart = alt.Chart(data).mark_line(point=True).encode(
#     x = alt.X('gw', scale=alt.Scale(domain=[1, latest_gw]), title="Gameweek"),
#     y=alt.Y('rank', aggregate={'argmax': 'gw'}, scale=alt.Scale(domain=[10,1]), title="Rank"),
#     color=alt.Color("team", legend=None),
# ).transform_window(
#     rank="rank()",
#     sort=[alt.SortField("rank", order="ascending")],
#     groupby=["gw"]
# ).properties(
#     title="Fake Internet Soccer XIV standings by gameweek",
#     width=700,
#     height=350,
# )

# labels = alt.Chart(data).mark_text(
#     align='left', dx=5
# ).encode(
#     x = alt.X('max(gw)', scale=alt.Scale(domain=[1, latest_gw])),
#     y=alt.Y('rank', aggregate={'argmax': 'gw'}, scale=alt.Scale(domain=[10,1])),
#     text='team:N',
#     color='team:N',
# ).transform_window(
#     rank="rank()",
#     sort=[alt.SortField("rank", order="ascending")],
#     groupby=["gw"]
# )

# chart + labels

In [7]:
df_trim.to_csv(f"data/output/standings/standings-season-{season}.csv", index=False)

In [8]:
latest_gw_df = df_trim[df_trim.gw == df_trim.gw.max()]

In [9]:
pivot_df = pd.pivot(df_trim, index='gw', columns='team', values='rank').reset_index()
pivot_df.to_csv(f'data/output/datawrapper/season-standings-{season}.csv', index=False)

In [10]:
season_charts = ensure_season_charts(season, template_season)
chart_id = season_charts['standings']['chart_id']

chart_url = f'{BASE_URL}/{chart_id}'
chart_data_url = f'{chart_url}/data'
json_headers = dw_headers()
csv_headers = dw_headers('text/csv')

In [11]:
update_data_response = requests.put(chart_data_url, data=pivot_df.to_csv(index=False), headers=csv_headers)
update_data_response

<Response [204]>

In [12]:
ranks_dict = latest_gw_df[['team','rank']].set_index('team')['rank'].to_dict()

In [13]:
chart_lines = {}

for team, rank in ranks_dict.items():
    symbol = "square" if rank % 2 == 0 else "circle"
    print(f"{team}: {symbol}")
    line_config = {
        "symbols": {
            "style": "hollow",
            "enabled": True,
            "shape": symbol
        }
    }

    chart_lines[team] = line_config

Benford FC: circle
West Thom United: square
Mattchester United: circle
NikkiPSG FC: square
ZacBurn Rovers FC: circle
Schadenasty's (Rafa): square
Newcastle UN-HINGED: circle
Seanhampton FC: square
Cheasle FC: circle
Thottenham Hotsluts: square


In [14]:
chart_lines

{'Benford FC': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'West Thom United': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}},
 'Mattchester United': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'NikkiPSG FC': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}},
 'ZacBurn Rovers FC': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 "Schadenasty's (Rafa)": {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}},
 'Newcastle UN-HINGED': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'Seanhampton FC': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}},
 'Cheasle FC': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'circle'}},
 'Thottenham Hotsluts': {'symbols': {'style': 'hollow',
   'enabled': True,
   'shape': 'square'}}}

In [15]:
# Team colors live in data/team-colors.json as {"Team name": "#hex"}, the same shape
# Datawrapper's color-category map takes. Every team name from every season is in there,
# so a new or renamed team just needs a line added.
# Colors not assigned yet are "TK" and are treated as missing.
team_colors = json.loads(Path('data/team-colors.json').read_text(encoding='utf-8'))
team_colors = {team: hex_ for team, hex_ in team_colors.items() if hex_.startswith('#')}
color_map = {team: team_colors[team] for team in ranks_dict if team in team_colors}

# Anyone missing gets a palette color nobody in this season is using, so the run still works
used = set(color_map.values())
spare = [h for h in dict.fromkeys(team_colors.values()) if h not in used]
for team in sorted(ranks_dict):
    if team not in color_map:
        color_map[team] = spare.pop(0) if spare else '#999999'
        print(f'{team!r} has no color in team-colors.json -> using {color_map[team]} for now')

payload = {
    "metadata": {
        "visualize": {
            "lines": chart_lines,
            "color-category": {"map": color_map},
        }
    }
}

In [16]:
patch_chart_response = requests.patch(chart_url, json=payload, headers=json_headers)
patch_chart_response

<Response [200]>

In [17]:
latest_chart_version = publish(chart_id, season, 'standings')

Published standings a1old v11
